# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [2]:
# Write your code below.
%load_ext dotenv
%dotenv


In [3]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [29]:
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")

parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)

dd_px = dd.read_parquet(parquet_files).set_index("ticker")

print(dd_px.head())



             Date   Open   High    Low  Close  Adj Close      Volume   source  \
ticker                                                                          
ACN    2001-07-19  15.10  15.29  15.00  15.17  11.404394  34994300.0  ACN.csv   
ACN    2001-07-20  15.05  15.05  14.80  15.01  11.284108   9238500.0  ACN.csv   
ACN    2001-07-23  15.00  15.01  14.55  15.00  11.276587   7501000.0  ACN.csv   
ACN    2001-07-24  14.95  14.97  14.70  14.86  11.171341   3537300.0  ACN.csv   
ACN    2001-07-25  14.70  14.95  14.65  14.95  11.238999   4208100.0  ACN.csv   

        Year  
ticker        
ACN     2001  
ACN     2001  
ACN     2001  
ACN     2001  
ACN     2001  


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
# Write your code below.
import pandas as pd

#calculate price for eaech stock
dd_shift = (  
         dd_px
        .groupby('ticker', group_keys=False)
        .apply(
                lambda x: (
                x.sort_values('Date', ascending = True)
                 .assign(Close_lag_1 = lambda df: df['Close'].shift(1))
                  .assign(Adj_Close_lag_1 = lambda df: df['Adj Close'].shift(1))
                ),
               meta = dd_px._meta.assign(Close_lag_1 = pd.Series(dtype = 'float64'), Adj_Close_lag_1 = pd.Series(dtype = 'float64')),
              )
        )

#returns and high low ratio
dd_feat = dd_shift.assign(
Returns = lambda x: x['Close']/x['Close_lag_1'] - 1,
high_low = lambda x: x['High']/x['Low'] - 1
)

print(dd_feat.head())

             Date   Open   High    Low  Close  Adj Close      Volume   source  \
ticker                                                                          
ACN    2001-07-19  15.10  15.29  15.00  15.17  11.404394  34994300.0  ACN.csv   
ACN    2001-07-20  15.05  15.05  14.80  15.01  11.284108   9238500.0  ACN.csv   
ACN    2001-07-23  15.00  15.01  14.55  15.00  11.276587   7501000.0  ACN.csv   
ACN    2001-07-24  14.95  14.97  14.70  14.86  11.171341   3537300.0  ACN.csv   
ACN    2001-07-25  14.70  14.95  14.65  14.95  11.238999   4208100.0  ACN.csv   

        Year  Close_lag_1  Adj_Close_lag_1   Returns  high_low  
ticker                                                          
ACN     2001          NaN              NaN       NaN  0.019333  
ACN     2001        15.17        11.404394 -0.010547  0.016892  
ACN     2001        15.01        11.284108 -0.000666  0.031615  
ACN     2001        15.00        11.276587 -0.009333  0.018367  
ACN     2001        14.86        11.171341

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [33]:
# Write your code below.
#convert dask dataframe to pandas dataframe
df_feat = dd_feat.compute()

#group by ticker, sort by date, and calculate 10-day rolling mean of returns
df_feat = (  
         df_feat
        .groupby('ticker', group_keys=False)
        .apply(
                lambda x: (
                x.sort_values('Date', ascending = True)
                 .assign(returns_10d = lambda df: df['Returns'].rolling(10).mean())
                ),
              )
        )
print(df_feat.head())


             Date   Open   High    Low  Close  Adj Close      Volume   source  \
ticker                                                                          
ACN    2001-07-19  15.10  15.29  15.00  15.17  11.404394  34994300.0  ACN.csv   
ACN    2001-07-20  15.05  15.05  14.80  15.01  11.284108   9238500.0  ACN.csv   
ACN    2001-07-23  15.00  15.01  14.55  15.00  11.276587   7501000.0  ACN.csv   
ACN    2001-07-24  14.95  14.97  14.70  14.86  11.171341   3537300.0  ACN.csv   
ACN    2001-07-25  14.70  14.95  14.65  14.95  11.238999   4208100.0  ACN.csv   

        Year  Close_lag_1  Adj_Close_lag_1   Returns  high_low  returns_10d  
ticker                                                                       
ACN     2001          NaN              NaN       NaN  0.019333          NaN  
ACN     2001        15.17        11.404394 -0.010547  0.016892          NaN  
ACN     2001        15.01        11.284108 -0.000666  0.031615          NaN  
ACN     2001        15.00        11.276587

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

In [ ]:
# Was it necessary to convert to pandas to calculate the moving average return?
# Dask also can do the work, converting to pandas is not necessary.



# Would it have been better to do it in Dask? Why?
# The course material mention start small: if possiblem use pandas，pandas is better for small data, and dask is better for large data. 
# if the data is small enough to fit in memory, pandas is faster and easier to use. 
# If the data is too large to fit in memory, dask is better.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.